In [166]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

NAMES = [name.strip() for name in open("names.txt", "r").readlines()]
CONTEXT_WIDTH = 3

DEVICE = "cpu"
#DEVICE = "mps"
#DEVICE = "cuda"

SEED = 42
RANDOM_GENERATOR = torch.Generator(device=DEVICE).manual_seed(SEED)

def name_to_tensor(name):
    return torch.tensor([0]+[1 + ord(c) - ord("a") for c in name]+[0])
def tensor_to_name(tensor):
    return "".join([chr(c + ord("a")-1) for c in tensor[1:-1]])

def sliding_mean(x: torch.Tensor, window: int) -> torch.Tensor:
    x = x.float()
    prefix = torch.cat((x.new_zeros(1), x.cumsum(dim=0)))
    return (prefix[window:] - prefix[:-window]) / window

def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | shape: {dt.shape == t.grad.shape} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

def prepareTestSets(names):
    X = []
    Y = []
    for name in names:
        t = name_to_tensor(name)
        x = [0] * CONTEXT_WIDTH

        for i in range(0, len(t)-1):
            x = x[1:] + [t[i].item()]
            X.append(x)
            Y.append(t[i+1].item())
    return (torch.tensor(X, device=DEVICE), torch.tensor(Y, device=DEVICE))

random.seed(SEED)
random.shuffle(NAMES)

names80p = int(len(NAMES)*0.8)
names90p = int(len(NAMES)*0.9)

inputX_train, inputY_train = prepareTestSets(NAMES[:names80p])
inputX_val, inputY_val = prepareTestSets(NAMES[names80p:names90p])
inputX_test, inputY_test = prepareTestSets(NAMES[names90p:])

In [397]:
VOCAB_SIZE = 27 # the number of characters in the vocabulary
EMBEDED_DIM = 10 # the dimensionality of the character embedding vectors
HIDDEN_LAYER_SIZE = 200 # the number of neurons in the hidden layer of the MLP
BLOCK_SIZE = 3 # the number of characters in the context window

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((VOCAB_SIZE, EMBEDED_DIM), generator=RANDOM_GENERATOR)
# Layer 1
W1 = torch.randn((EMBEDED_DIM * BLOCK_SIZE, HIDDEN_LAYER_SIZE), generator=RANDOM_GENERATOR) * (5/3)/((EMBEDED_DIM * BLOCK_SIZE)**0.5)
b1 = torch.randn(HIDDEN_LAYER_SIZE, generator=RANDOM_GENERATOR) * 0.1 # using b1 just for fun, it's useless because of BN
# Layer 2
W2 = torch.randn((HIDDEN_LAYER_SIZE, VOCAB_SIZE), generator=RANDOM_GENERATOR) * 0.1
b2 = torch.randn(VOCAB_SIZE, generator=RANDOM_GENERATOR) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, HIDDEN_LAYER_SIZE))*0.1 + 1.0
bnbias = torch.randn((1, HIDDEN_LAYER_SIZE))*0.1

# Note: I am initializating many of these parameters in non-standard ways
# because sometimes initializating with e.g. all zeros could mask an incorrect
# implementation of the backward pass.

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

12297


In [362]:
# Prepare minibatches
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, inputX_train.shape[0], (batch_size,), generator=g)
Xb, Yb = inputX_train[ix], inputY_train[ix] # batch X,Y



In [364]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation
# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5

bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact) # hidden layer
# Linear layer 2
logits = h @ W2 + b2 # output layer
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.3806, grad_fn=<NegBackward0>)

In [365]:
# Exercise 1: backprop through the whole thing manually, 
# backpropagating through exactly all of the variables 
# as they are defined in the forward pass above, one by one


dlogprobs = -torch.stack([F.one_hot(Yb[b], VOCAB_SIZE) for b in range(n)]) / n
dprobs = dlogprobs * (1.0 / probs)
dcounts_sum_inv = (dprobs * counts).sum(1, keepdims=True)
dcounts_sum = dcounts_sum_inv * (-counts_sum**-2)
dcount = torch.ones_like(counts) * (dcounts_sum  + dprobs * counts_sum_inv)
dnorm_logits = torch.exp(norm_logits) * dcount
dlogit_maxes = -dnorm_logits.sum(1, keepdims=True)
dlogits = dnorm_logits.clone() + F.one_hot(logits.max(1).indices, VOCAB_SIZE) * dlogit_maxes

dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
dhpreact = dh * (1 - torch.tanh(hpreact)**2)
dbngain = (dhpreact * bnraw).sum(0, keepdim=True)
dbnbias = (dhpreact).sum(0, keepdim=True)
dbnraw = dhpreact * bngain
dbnvar_inv = (dbnraw * bndiff).sum(0, keepdim=True)
dbnvar = dbnvar_inv * (-0.5*(bnvar + 1e-5)**-1.5)
dbndiff2 = torch.ones_like(bndiff2) * 1/(n-1) * dbnvar
dbndiff = 2.0 * bndiff * dbndiff2 + dbnraw * bnvar_inv
dbnmeani = -1.0 * dbndiff.sum(0, keepdim=True)
dhprebn = dbndiff + torch.ones_like(hprebn) * 1/n * dbnmeani
dW1 = (embcat.T @ dhprebn)
db1 = dhprebn.sum(0)
dembcat = dhprebn @ W1.T
demb = dembcat.view_as(emb)
#dC = torch.stack([F.one_hot(Xb[i,j], VOCAB_SIZE).view(-1,1) * demb[i,j] for j in range(Xb.shape[1]) for i in range(Xb.shape[0])]).sum(0)
dC = torch.zeros_like(C)
for i in range(Xb.shape[0]):
  for j in range(Xb.shape[1]):
    dC[Xb[i,j]] += demb[i,j]

cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcount, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)#
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('embcat', dembcat, embcat)
cmp('emb', demb, emb)
cmp('C', dC, C)







logprobs        | shape: True | exact: True  | approximate: True  | maxdiff: 0.0
probs           | shape: True | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | shape: True | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | shape: True | exact: True  | approximate: True  | maxdiff: 0.0
counts          | shape: True | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | shape: True | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | shape: True | exact: True  | approximate: True  | maxdiff: 0.0
logits          | shape: True | exact: True  | approximate: True  | maxdiff: 0.0
h               | shape: True | exact: True  | approximate: True  | maxdiff: 0.0
W2              | shape: True | exact: True  | approximate: True  | maxdiff: 0.0
b2              | shape: True | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | shape: True | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | shape: Tru

In [366]:
# Exercise 2: backprop through cross_entropy but all in one go
# to complete this challenge look at the mathematical expression of the loss,
# take the derivative, simplify the expression, and just write it out

# forward pass

# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())


#backward_pass

dlogits = probs.clone()
dlogits[range(n), Yb] -= 1.0
dlogits *= 1/n

cmp('logits', dlogits, logits) # I can only get approximate to be true, my maxdiff is 6e-9

3.3806042671203613 diff: 0.0
logits          | shape: True | exact: False | approximate: True  | maxdiff: 5.122274160385132e-09


In [367]:
# Exercise 3: backprop through batchnorm but all in one go
# to complete this challenge look at the mathematical expression of the output of batchnorm,
# take the derivative w.r.t. its input, simplify the expression, and just write it out

# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias


# now:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

# backward pass

# before we had:
# dbnraw = bngain * dhpreact
# dbndiff = bnvar_inv * dbnraw
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
# dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
# dbndiff += (2*bndiff) * dbndiff2
# dhprebn = dbndiff.clone()
# dbnmeani = (-dbndiff).sum(0)
# dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)

# calculate dhprebn given dhpreact (i.e. backprop through the batchnorm)
# (you'll also need to use some of the variables from the forward pass up above)

psi = bnvar_inv**-1
t_first = dhpreact / psi 
t_second = dhpreact.sum(0, keepdim=True) / (n*psi)
t_third = bndiff * (dhpreact * bndiff / ((n-1)*psi**3)).sum(0, keepdim=True)
dhprebn = (t_first - t_second - t_third) * bngain
print(dhprebn.shape)

dhprebn2 =  bngain * bnvar_inv * (dhpreact - (1/n) * dhpreact.sum(0) - (bnraw/(n-1)) * (dhpreact * bnraw).sum(0))

cmp('hprebn', dhprebn, hprebn) # I can only get approximate to be true, my maxdiff is 9e-10
cmp('hprebn', dhprebn2, hprebn)

max diff: tensor(4.7684e-07, grad_fn=<MaxBackward1>)
torch.Size([32, 64])
hprebn          | shape: True | exact: False | approximate: True  | maxdiff: 9.313225746154785e-10
hprebn          | shape: True | exact: False | approximate: True  | maxdiff: 9.313225746154785e-10


In [486]:
#parameters = [C, W1, b1, W2, b2, bngain, bnbias]

def go(X, Y, iterations=100, learning_rate=0.1, print_every=1000):
    for it in range(iterations):
        ix = torch.randint(0, X.shape[0], (batch_size,), generator=RANDOM_GENERATOR)
        Xb = X[ix]
        Yb = Y[ix]
        
        emb = C[Xb] # embed the characters into vectors
        embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
        hprebn = embcat @ W1 + b1 # hidden layer pre-activation
        
        bnvar_inv = 1.0/torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5)
        bnraw = (hprebn - hprebn.mean(0, keepdim=True)) * bnvar_inv
        hpreact = bngain * bnraw + bnbias
        h = torch.tanh(hpreact) # hidden layer
        logits = h @ W2 + b2 # output layer
        probs = F.softmax(logits, dim=1)
        loss = F.cross_entropy(logits, Yb)

        for p in parameters:
            p.grad = None
        loss.backward()

        # backward pass
        dlogits = probs.clone()
        dlogits[range(n), Yb] -= 1.0
        dlogits *= 1/n
        dh = dlogits @ W2.T
        dW2 = h.T @ dlogits
        db2 = dlogits.sum(0)

        dhpreact = dh * (1 - torch.tanh(hpreact)**2)
        dhprebn = bngain * bnvar_inv * (dhpreact - (1/n) * dhpreact.sum(0) - (bnraw/(n-1)) * (dhpreact * bnraw).sum(0))

        dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
        dbnbias = dhpreact.sum(0, keepdim=True)

        dW1 = (embcat.T @ dhprebn)
        db1 = dhprebn.sum(0)
        dembcat = dhprebn @ W1.T
        demb = dembcat.view_as(emb)
        dC = torch.zeros_like(C)
        for i in range(Xb.shape[0]):
            for j in range(Xb.shape[1]):
                dC[Xb[i,j]] += demb[i,j]
        grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]

        if it%print_every == 0:
            print(it, f'loss: {loss.item()}')

        with torch.no_grad():
            for p, g in zip(parameters, grads):
                p.data -= learning_rate * g

    return (loss, parameters, grads)

(loss, params, grads) = go(inputX_train, inputY_train, iterations=20000, learning_rate=0.01, print_every=5000)
for p, g in zip(params, grads):
    cmp(str(tuple(p.shape)), g, p)


0 loss: 2.089310884475708
5000 loss: 2.195514678955078
10000 loss: 2.1824276447296143
15000 loss: 2.423336982727051
(27, 10)        | shape: True | exact: False | approximate: False | maxdiff: 2.9802322387695312e-08
(30, 200)       | shape: True | exact: False | approximate: True  | maxdiff: 1.1175870895385742e-08
(200,)          | shape: True | exact: False | approximate: True  | maxdiff: 7.450580596923828e-09
(200, 27)       | shape: True | exact: False | approximate: True  | maxdiff: 2.2351741790771484e-08
(27,)           | shape: True | exact: False | approximate: True  | maxdiff: 7.450580596923828e-09
(1, 200)        | shape: True | exact: False | approximate: True  | maxdiff: 3.725290298461914e-09
(1, 200)        | shape: True | exact: False | approximate: True  | maxdiff: 7.450580596923828e-09


In [487]:
run_bnmean=[]
run_bnvar=[]
with torch.no_grad():
  # pass the training set through
  emb = C[inputX_train]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 + b1
  # measure the mean/std over the entire training set
  run_bnmean = hpreact.mean(0, keepdim=True)
  run_bnvar = hpreact.var(0, keepdim=True, unbiased=True)

In [489]:
@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (inputX_train, inputY_train),
    'val': (inputX_val, inputY_val),
    'test': (inputX_test, inputY_test),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 + b1
  hpreact = bngain * (hpreact - run_bnmean) * (run_bnvar + 1e-5)**-0.5 + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')
split_loss('test')

train 2.0756332874298096
val 2.1165213584899902
test 2.1131110191345215


In [ ]:
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * 3 # initialize with all ...
    while True:
      # ------------
      # forward pass:
      # Embedding
      emb = C[torch.tensor([context])] # (1,block_size,d)      
      embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
      hpreact = embcat @ W1 + b1
      hpreact = bngain * (hpreact - run_bnmean) * (run_bnvar + 1e-5)**-0.5 + bnbias
      h = torch.tanh(hpreact) # (N, n_hidden)
      logits = h @ W2 + b2 # (N, vocab_size)
      # ------------
      # Sample
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))